# Chapter 3: Essential Math for QC

---

Quantum computing is, at its core, a linear-algebra machine over the complex numbers: every quantum state is a vector, every operation is a matrix, and every measurement is an eigenvalue problem *(Book §3, intro)*. This notebook builds that foundation with short, runnable NumPy examples — complex numbers, inner products, outer products and reflection operators, tensor products, conjugate transposes, determinants and inverses, eigenvalue decomposition, matrix polynomials and exponentials, the SVD, and the Gershgorin bound.

The chapter deliberately treats **two** matrix families side by side: the real-valued symmetric matrices arising from engineering discretizations (truss, Poisson, plane stress) and the complex-valued Hermitian and unitary matrices of quantum mechanics. They share one toolkit under the mapping *Real → Complex, Transpose → Conjugate transpose, Symmetric → Hermitian, Orthogonal → Unitary*.

**Prerequisites:**
- See `Chapter02_QuantumSoftware_notebook.ipynb` for installation instructions

In [ ]:
# Setup and imports
import numpy as np
import matplotlib.pyplot as plt


## Creating complex numbers  *(Book §3.2, Example 3.1, Listing 3.1)*
Quantum amplitudes are complex numbers, and complex interference is the mechanism behind most quantum speedups. For $x = a + bi$, the real part is $a$, the imaginary part is $b$, the conjugate is $x^{*}=a-bi$, and the magnitude is $|x|=\sqrt{xx^{*}}=\sqrt{a^2+b^2}$. In Python the symbol `j` denotes $i=\sqrt{-1}$ (note the literal `3j`); we reserve `j` for this purpose and avoid it as a loop index. Here $x=1+3i$, so $x^{*}=1-3i$ and $|x|=\sqrt{10}$.

In [ ]:


j = np.emath.sqrt(-1) # define j as sqrt(-1) for complex numbers
#%% Complex variables
x = 1 + 3j # note the 3j
print("The real part is: ", x.real)
print("The imaginary part is: ", x.imag)
print("The absolute value is: ", abs(x))

## Inner product  *(Book §3.3, Definitions 3.1–3.4, Example 3.4, Listing 3.2)*
The inner product of two vectors is $\langle u,v\rangle \equiv u^{\dagger}v = \sum_i u_i^{*}\,v_i$, built on the conjugate transpose $u^{\dagger}$. It is a complex number, and in general $\langle u,v\rangle=\langle v,u\rangle^{*}$. The two vectors here, $v_0=\tfrac{1}{\sqrt2}\begin{bmatrix}1\\1\end{bmatrix}$ and $v_1=\tfrac{1}{\sqrt2}\begin{bmatrix}1\\-1\end{bmatrix}$, are **orthonormal**: each has unit norm and $\langle v_0,v_1\rangle=0$. Orthonormal vectors form the natural basis for quantum states (Chapter 7).

In [ ]:
v0 = np.array([1/np.sqrt(2), 1/np.sqrt(2)])
v1 = np.array([1/np.sqrt(2), -1/np.sqrt(2)])

# Inner product using np.vdot (handles conjugation)
print("Inner product <v0,v0>:", np.vdot(v0, v0))
print("Inner product <v0,v1>:", np.vdot(v0, v1))

## Outer products and reflection operators  *(Book §3.4, Definitions 3.6–3.7, Examples 3.5–3.6, Listing 3.3)*
The **inner product** $u^{\dagger}v$ combines two vectors into a *scalar*; reversing the order gives the **outer product** $u\,v^{\dagger}$, which is a *matrix* with entries $(u\,v^{\dagger})_{ij}=u_i v_j^{*}$. This is our tool for building operators directly from vectors. A key special case is the outer product of a **unit** vector with itself, $P=v\,v^{\dagger}$, the **projector** onto $v$ — it is Hermitian and idempotent ($P^2=P$). From it we form the **reflection** $R=I-2\,v v^{\dagger}$, which is both unitary and Hermitian with $R^2=I$: it flips the component along $v$ and leaves everything orthogonal to $v$ unchanged. This reflection is exactly the building block of Grover's search algorithm (Chapter 12). Note `np.outer` does *not* conjugate its second argument, so we pass `v.conj()` to get $v\,v^{\dagger}$.

In [ ]:
np.set_printoptions(precision=3, suppress=True)  # tidy display of tiny fp values
# Outer product u v^dag (np.outer does not conjugate; pass v.conj())
u = np.array([1, 1j])
v = np.array([1, -1j])
print("u v^dag = \n", np.outer(u, v.conj()))

# Projector onto a unit vector w:  P = w w^dag  (Hermitian, idempotent)
w = (1/np.sqrt(2)) * np.array([1.0, 1.0])
P = np.outer(w, w.conj())
print("P = w w^dag = \n", P)
print("P^2 = P ?", np.allclose(P @ P, P))

# Reflection about the hyperplane orthogonal to w:  R = I - 2P
# (unitary and Hermitian, with R^2 = I) -- the Grover building block
R = np.eye(2) - 2*P
print("R = I - 2P = \n", R)
print("R unitary?", np.allclose(R.conj().T @ R, np.eye(2)))
print("R^2 = I ?", np.allclose(R @ R, np.eye(2)))

## Tensor product of vectors  *(Book §3.5, Listing 3.4)*
The tensor product $\otimes$ combines two vectors into a higher-dimensional one; for two 2-vectors it produces a 4-vector of all pairwise products. NumPy's `np.kron` implements it. Here $v_1=|0\rangle=\begin{bmatrix}1\\0\end{bmatrix}$ and $v_2=|1\rangle=\begin{bmatrix}0\\1\end{bmatrix}$, so $v_1\otimes v_2 = |01\rangle = \begin{bmatrix}0\\1\\0\\0\end{bmatrix}$ — the building block of multi-qubit states.

In [ ]:
v1 = np.array([1, 0])
v2 = np.array([0, 1])
v3 = np.kron(v1, v2)
print("v3 = \n", v3)

## Conjugate transpose of a matrix  *(Book §3.6, Definitions 3.8–3.10)*
The conjugate transpose $A^{\dagger}$ is the transpose of the complex conjugate of $A$ — the complex generalization of the ordinary transpose. A matrix is **Hermitian** if $A=A^{\dagger}$ (the complex analogue of symmetric), and **unitary** if $A^{\dagger}A=I$ (the analogue of orthogonal). Here we form $A^{T}$ and $A^{\dagger}$ for a complex matrix and compare them.

In [ ]:
A = np.array([[1+1j, 2], [0, 3]])
print("A = \n", A)

# Transpose
A_T = A.T
print("A^T = \n", A_T)

A_dag = np.conjugate(A).T  
print("A^dag = \n", A_dag)

## Determinant and inverse  *(Book §3.6, Listing 3.5)*
A square matrix is invertible only if it is **non-singular**, i.e. its determinant is nonzero, and then $A^{-1}$ satisfies $AA^{-1}=I$. Direct methods (LU) cost $O(N^3)$ operations and $O(N^2)$ memory, so for large systems iterative solvers are preferred. Here we compute $\det(A)$, form $A^{-1}$, and verify $A\,A^{-1}=I$.

In [ ]:

# Define matrix A
A = np.array([[1+1j, 2], [0, 3]])

# Check determinant (non-zero for invertible matrix)
det_A = np.linalg.det(A)
print("det(A) = ", det_A)

# Compute inverse
A_inv = np.linalg.inv(A)
print("A^{-1} = \n", A_inv)

# Verify A * A^{-1} = I
I_check = A @ A_inv
print("A * A^{-1} = \n", I_check)



## Tensor product of matrices  *(Book §3.6, subsection "Tensor product of matrices")*
The construction used for vectors extends to matrices: $A\otimes B$ replaces each entry of $A$ with that scalar multiple of the whole matrix $B$, so an $M\times N$ matrix tensored with a $P\times Q$ matrix gives an $(MP)\times(NQ)$ block matrix.

Two special cases carry the weight later. $I_N\otimes A$ is **block-diagonal** — $N$ independent, non-interacting copies of $A$ — while $A\otimes I_N$ is *not*, since each entry of $A$ becomes a scaled identity block instead. The order matters, and both forms appear when stiffness matrices are written compactly for quantum encoding (Chapter 4). The last block shows the other role of the tensor product: repeatedly tensoring single-qubit ($2\times2$) operators is what builds the $2^n$-dimensional operators of an $n$-qubit system, and unitarity survives every step.

In [ ]:
np.set_printoptions(precision=3, suppress=True)

# --- the 2x2 (x) 2x2 pattern from the book -------------------------------
a, b, c, d = 1, 2, 3, 4
e, f, g, h = 5, 6, 7, 8
A = np.array([[a, b], [c, d]])
B = np.array([[e, f], [g, h]])
book = np.array([[a*e, a*f, b*e, b*f],       # each entry of A scaled by all of B
                 [a*g, a*h, b*g, b*h],
                 [c*e, c*f, d*e, d*f],
                 [c*g, c*h, d*g, d*h]])
print("A (x) B =\n", np.kron(A, B))
print("matches the book's 4x4 expansion?", np.array_equal(np.kron(A, B), book))

# --- general shapes:  (M x N) (x) (P x Q)  ->  (MP x NQ) ------------------
for (M, N), (P, Q) in [((2, 3), (2, 2)), ((3, 1), (2, 4))]:
    X, Y = np.ones((M, N)), np.ones((P, Q))
    print(f"({M}x{N}) (x) ({P}x{Q})  ->  {np.kron(X, Y).shape}   expected ({M*P}x{N*Q})")

# --- Case 1: I_N (x) A  is block diagonal ---------------------------------
from scipy.linalg import block_diag
A = np.array([[2., -1.], [-1., 2.]])
K1 = np.kron(np.eye(3), A) + 0.0
print("\nI_3 (x) A =\n", K1)
print("three independent copies of A on the diagonal?",
      np.allclose(K1, block_diag(A, A, A)))

# --- Case 2: A (x) I_N  is NOT block diagonal -----------------------------
K2 = np.kron(A, np.eye(3)) + 0.0
print("\nA (x) I_3 =\n", K2)
print("block-diagonal? (is the off-diagonal 3x3 block zero?)", np.allclose(K2[:3, 3:], 0))
print("same as I_3 (x) A ?", np.allclose(K1, K2), " -- the tensor product does not commute")

# --- the 2^n structure:  n qubits from n single-qubit spaces --------------
H = np.array([[1., 1.], [1., -1.]]) / np.sqrt(2)
Hn = H
for n in range(2, 4):
    Hn = np.kron(Hn, H)
    print(f"\nH^(x){n}: shape {Hn.shape} = 2^{n} x 2^{n}   still unitary?",
          np.allclose(Hn.conj().T @ Hn, np.eye(2**n)))

## Eigenvalue decomposition  *(Book §3.7, Listing 3.6)*
Eigen-pairs satisfy $A v_k=\lambda_k v_k$, and a diagonalizable matrix factors as $A=VDV^{-1}$ with the eigenvectors as columns of $V$ and the eigenvalues on the diagonal of $D$. For a **symmetric** matrix the eigenvalues are real and $V$ is orthogonal, giving the compact form $A=VDV^{\dagger}$. For $A=\begin{bmatrix}1.5&0.5\\0.5&1.5\end{bmatrix}$ the eigenvalues are $\lambda_0=1$ and $\lambda_1=2$. This decomposition is the key to computing matrix functions efficiently.

In [ ]:
A = np.array([[1.5, 0.5],
              [0.5, 1.5]])

eigenvalues, eigenvectors = np.linalg.eig(A)

print("\nEigenvalues:")
print(eigenvalues)
print("\nEigenvectors (as columns):")
print(eigenvectors)

## Matrix polynomials  *(Book §3.8, Definition 3.12, Theorem 3.1, Example 3.11, Listing 3.10)*
For a polynomial $P(x)=a_0+a_1x+\cdots+a_nx^n$, the **matrix polynomial** is $P(A)=a_0I+a_1A+\cdots+a_nA^n$. The useful fact is the **Matrix Polynomial Theorem**: if $A=VDV^{\dagger}$ then $P(A)=V\,P(D)\,V^{\dagger}$, where $P(D)=\mathrm{diag}\big(P(\lambda_0),\ldots,P(\lambda_{N-1})\big)$. In words: **applying a polynomial to a matrix applies it to each eigenvalue**, leaving the eigenvectors untouched. That is why $P(A)$ can be evaluated without ever forming large matrix powers — and it is the mechanism QSVT exploits (Chapter 19), where a quantum circuit applies a polynomial to the *singular* values of a matrix.

Below we evaluate $P(A)=I+2A+A^2$ two ways for the book's $A=\begin{bmatrix}2&-1\\-1&2\end{bmatrix}$ (eigenvalues $\lambda_0=1$, $\lambda_1=3$) and confirm both give $\begin{bmatrix}10&-6\\-6&10\end{bmatrix}$.

In [ ]:
np.set_printoptions(precision=3, suppress=True)

# --- Book Listing 3.10: p(A) = I + 2A + A^2 -------------------------------
A = np.array([[1, 1], [0, 2]])
I = np.eye(2)
p_A = I + 2*A + A @ A
print("A =\n", A)
print("p(A) = I + 2A + A^2 =\n", p_A)

# Eigenvalue property: p(A) has eigenvalues p(lambda_k)
lam, _ = np.linalg.eig(A)
print("\neigenvalues of A      :", np.sort(lam))
print("p(lambda) for each    :", np.sort(1 + 2*lam + lam**2))
print("eigenvalues of p(A)   :", np.sort(np.linalg.eigvals(p_A)))

# --- Book Example 3.11, both routes ---------------------------------------
A = np.array([[2., -1.], [-1., 2.]])          # eigenvalues 1 and 3
P = lambda x: 1 + 2*x + x**2

direct = np.eye(2) + 2*A + A @ A
print("\nExample 3.11, direct   : P(A) =\n", direct)

lam, V = np.linalg.eig(A)
idx = np.argsort(lam)                          # book convention: ascending
lam, V = lam[idx], V[:, idx]
viaEVD = V @ np.diag(P(lam)) @ V.conj().T      # Theorem 3.1
print("Example 3.11, Thm 3.1  : V P(D) V^dag =\n", viaEVD)
print("lambda_0, lambda_1 =", lam, " ->  P(1), P(3) =", P(lam))
print("both routes agree?", np.allclose(direct, viaEVD))

## Matrix exponential  *(Book §3.9, Definition 3.13, Theorem 3.3, Example 3.14, Listing 3.11)*
The matrix exponential is the same Taylor series with a matrix argument, $e^{A}=\sum_k A^k/k!$, and by the same eigenvalue argument $e^{A}=V e^{D} V^{\dagger}$ with $e^{D}$ taken element-wise on the diagonal. The result the rest of the book leans on is **Theorem 3.3: if $A$ is Hermitian, then $e^{iA}$ is unitary** — this is why quantum gates can be written as matrix exponentials, and why $e^{-iAt}$ (quantum time evolution) is a legal operation.

We check three things: that $e^{i\theta Y}=I\cos\theta+iY\sin\theta$ is the real rotation matrix $\begin{bmatrix}\cos\theta&\sin\theta\\-\sin\theta&\cos\theta\end{bmatrix}$ (Example 3.14); that the Hermitian hypothesis in Theorem 3.3 is doing real work; and Exercise 3.23, $e^{i\pi A/2}$. Note `scipy.linalg.expm` computes the true matrix exponential — `np.exp` would exponentiate element-wise, which is a different and wrong thing.

In [ ]:
from scipy.linalg import expm

# --- Book Listing 3.11: e^D is element-wise on the diagonal ---------------
D = np.diag([1., 2.])
print("exp(D) =\n", expm(D), "   (e^1 =", round(np.e, 3), ", e^2 =", round(np.e**2, 3), ")")

# --- Book Example 3.14:  e^{i theta Y} = I cos(theta) + i Y sin(theta) ----
Y = np.array([[0, -1j], [1j, 0]])
theta = np.pi/4

U_expm    = expm(1j*theta*Y)
U_formula = np.eye(2)*np.cos(theta) + 1j*Y*np.sin(theta)
U_rot     = np.array([[np.cos(theta),  np.sin(theta)],
                      [-np.sin(theta), np.cos(theta)]])
print("\nexp(i.theta.Y) via expm =\n", U_expm)
print("I cos+ iY sin          =\n", U_formula)
print("rotation matrix form   =\n", U_rot)
print("all three agree?", np.allclose(U_expm, U_formula) and np.allclose(U_expm, U_rot))
print("purely real?", np.allclose(U_expm.imag, 0), "| unitary?",
      np.allclose(U_expm.conj().T @ U_expm, np.eye(2)))

# --- Theorem 3.3: A Hermitian  =>  e^{iA} unitary -------------------------
A = np.array([[1, 1j], [-1j, 2]])                       # Hermitian
print("\nA Hermitian?", np.allclose(A, A.conj().T),
      "| e^{iA} unitary?", np.allclose(expm(1j*A).conj().T @ expm(1j*A), np.eye(2)))

B = np.array([[0., 1.], [0., 0.]])                      # NOT Hermitian
print("B Hermitian?", np.allclose(B, B.conj().T),
      "| e^{iB} unitary?", np.allclose(expm(1j*B).conj().T @ expm(1j*B), np.eye(2)))

# --- Book Exercise 3.23:  e^{i pi A / 2} for A = [[2,-1],[-1,2]] ----------
A = np.array([[2., -1.], [-1., 2.]])
print("\nexp(i.pi.A/2) =\n", expm(1j*np.pi*A/2), "  (book answer: [[0, i], [i, 0]])")

## When eigenvalue decomposition fails  *(Book §3.7; this matrix returns as Example 3.18 in §3.10, Singular Value Decomposition)*
Not every matrix is diagonalizable. The matrix $A=\begin{bmatrix}2&1\\0&2\end{bmatrix}$ is **defective**: it has the repeated eigenvalue $2$ but only a *single* independent eigenvector, so no full set of eigenvectors exists and $A=VDV^{-1}$ cannot be formed. This is why the chapter stresses that in quantum computing we work almost exclusively with **Hermitian and unitary** matrices — both are *always* guaranteed diagonalizable with orthonormal eigenvectors. It is also why the chapter turns to the **SVD** (§3.10), which exists for *every* matrix — including this one.

In [ ]:
A = np.array([[2, 1], [0, 2]])
D, V = np.linalg.eig(A)
print("Eigenvalues of A:", D)
print("Eigenvectors of A (columns of V):\n", V)

print("A = \n", V @ np.diag(D) @ np.linalg.inv(V), "?")  

## Singular value decomposition  *(Book §3.10, Examples 3.15–3.18)*
The SVD factors **any** matrix — square or rectangular, diagonalizable or not — as $A = L\,\Sigma\,R^{\dagger}$ with $L$ ($M\times M$) and $R$ ($N\times N$) unitary and $\Sigma$ ($M\times N$) diagonal with non-negative entries $\sigma_1\ge\sigma_2\ge\cdots\ge 0$. Where the eigenvalue decomposition needs a square, diagonalizable matrix, the SVD always exists — which is why QSVT (Chapter 19) is built on singular values rather than eigenvalues.

Two conventions to watch. **NumPy's third return value is $R^{\dagger}$, not $R$**, and it returns $\Sigma$ as a 1-D array of singular values, so we rebuild the matrix explicitly. And singular values come back **descending**, whereas this book orders eigenvalues **ascending** — so for an SPD matrix, where the two sets coincide, they appear in opposite order (Example 3.16 below).

In [ ]:
np.set_printoptions(precision=4, suppress=True)

def show_svd(A, name, book=None):
    L, s, Rh = np.linalg.svd(A)                 # note: third output is R^dag, not R
    Sig = np.zeros_like(A, dtype=float); np.fill_diagonal(Sig, s)
    print(f"{name}\n  sigma  = {np.round(s,4)}")
    print("  L      =\n", L)
    print("  R^dag  =\n", Rh)
    print("  L Sigma R^dag == A ?", np.allclose(L @ Sig @ Rh, A))
    if book is not None:
        print("  matches book values?", np.allclose(book[0] @ np.diag(book[1]) @ book[2], A, atol=5e-4))
    print()

# Example 3.15 -- rank-deficient: one singular value is exactly 0
show_svd(np.array([[0.1, 0.2], [0.2, 0.4]]), "Example 3.15  A = [[0.1,0.2],[0.2,0.4]]")

# Example 3.16 -- SPD: singular values coincide with eigenvalues (reverse order)
A = np.array([[2., 0., 1.], [0., 2., 0.], [1., 0., 2.]]) / 3
print("Example 3.16  (SPD)")
print("  singular values (descending):", np.round(np.linalg.svd(A, compute_uv=False), 4))
print("  eigenvalues     (ascending) :", np.round(np.sort(np.linalg.eigvalsh(A)), 4))
print("  same set, opposite order.\n")

# Example 3.17 -- rectangular 2x3
show_svd(np.array([[1., 2., 0.], [0., 1., 3.]]), "Example 3.17  A (2x3)")

# Example 3.18 -- not diagonalizable, but the SVD still exists
A = np.array([[2., 1.], [0., 2.]])
show_svd(A, "Example 3.18  A = [[2,1],[0,2]]  (defective)",
         book=(np.array([[0.7882, -0.6154], [0.6154, 0.7882]]),
               np.array([2.5616, 1.5616]),
               np.array([[0.6154, 0.7882], [-0.7882, 0.6154]])))
print("  closed form  sigma = (sqrt(17) +/- 1)/2 :",
      np.round([(np.sqrt(17)+1)/2, (np.sqrt(17)-1)/2], 4),
      "|  sigma_1 * sigma_2 = det(A) =", round(np.linalg.det(A), 4))

## Gershgorin's theorem  *(Book §3.11, Example 3.19)*
Several quantum algorithms (Quantum Phase Estimation, HHL) need cheap bounds on a matrix's eigenvalues without computing them. Gershgorin's theorem provides exactly that: for each row $i$, form a disc in the complex plane centered at the diagonal entry $a_{ii}$ with radius $R_i=\sum_{j\ne i}|a_{ij}|$; every eigenvalue then lies in the union of these discs. Here we build the discs for a $3\times3$ complex matrix and confirm the computed eigenvalues fall inside them.

In [ ]:

# Define the matrix
A = np.array([[4+2j, 1, 0.5j],
              [0.5, 3, 1-1j],
              [0.2+0.3j, 0.5j, 1+1j]])

# Compute eigenvalues
eigenvalues = np.linalg.eigvals(A)

# Calculate Gershgorin discs
centers = np.diag(A)
radii = np.array([
    np.sum(np.abs(A[0, :])) - np.abs(A[0, 0]),
    np.sum(np.abs(A[1, :])) - np.abs(A[1, 1]),
    np.sum(np.abs(A[2, :])) - np.abs(A[2, 2])
])

print("Matrix A:")
print(A)
print("\nEigenvalues:")
for i, eig in enumerate(eigenvalues):
    print(f"λ_{i+1} = {eig:.4f}")

print("\nGershgorin Discs:")
for i in range(3):
    print(f"Disc {i+1}: Center = {centers[i]:.4f}, Radius = {radii[i]:.4f}")

# Verify eigenvalues are in discs
print("\nVerification:")
for i, eig in enumerate(eigenvalues):
    in_disc = []
    for j in range(3):
        dist = np.abs(eig - centers[j])
        if dist <= radii[j]:
            in_disc.append(j+1)
    print(f"λ_{i+1} is in disc(s): {in_disc}")

# Visualization
fig, ax = plt.subplots(figsize=(10, 8))

# Plot Gershgorin discs
colors = ['blue', 'red', 'green']
for i in range(3):
    circle = plt.Circle((centers[i].real, centers[i].imag), 
                        radii[i], color=colors[i], 
                        alpha=0.2, label=f'Disc {i+1}')
    ax.add_patch(circle)
    ax.plot(centers[i].real, centers[i].imag, 'o', 
            color=colors[i], markersize=8)

# Plot eigenvalues
ax.plot(eigenvalues.real, eigenvalues.imag, 'kx', 
        markersize=12, markeredgewidth=3, label='Eigenvalues')

# Formatting
ax.set_xlabel('Real', fontsize=12)
ax.set_ylabel('Imaginary', fontsize=12)
ax.set_title('Gershgorin Circle Theorem', fontsize=14)
ax.grid(True, alpha=0.3)
ax.axis('equal')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()
